# Neurosynth Coordinate Decoder

Given a list of MNI coordinates, query the Neurosynth database (v7) for the most strongly associated cognitive terms.

Uses an association test: z-score comparing term frequency in studies activating near the seed coordinate vs. all studies in the database.

## Setup

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
from IPython.display import display

# Where to store / look for the Neurosynth v7 data files
NS_DATA_DIR = Path.home() / 'data' / 'ns_data' / 'neurosynth'

# Search radius in mm around each coordinate
RADIUS_MM = 10

# Number of top terms to display per coordinate
N_TOP_TERMS = 15

## Load Neurosynth data

In [5]:
from nimare.extract import fetch_neurosynth

# NS_DATA_DIR.mkdir(parents=True, exist_ok=True)
# fetch_neurosynth(str(NS_DATA_DIR.parent), version='7', overwrite=False)

print('Loading coordinates...')
coords_df = pd.read_csv(NS_DATA_DIR / 'data-neurosynth_version-7_coordinates.tsv.gz', sep='\t')
study_ids = sorted(coords_df['id'].unique())
id_to_idx = {sid: i for i, sid in enumerate(study_ids)}
print(f'  {len(coords_df)} peaks across {len(study_ids)} studies')

print('Loading vocabulary...')
with open(NS_DATA_DIR / 'data-neurosynth_version-7_vocab-terms_vocabulary.txt') as f:
    vocab = [line.strip() for line in f]
print(f'  {len(vocab)} terms')

print('Loading term features...')
features = sp.load_npz(NS_DATA_DIR / 'data-neurosynth_version-7_vocab-terms_source-abstract_type-tfidf_features.npz')
feat_bin = (features > 0).toarray()
p_term_global = feat_bin.mean(axis=0)
print(f'  feature matrix: {features.shape}')

Loading coordinates...
  507891 peaks across 14371 studies
Loading vocabulary...
  3228 terms
Loading term features...
  feature matrix: (14371, 3228)


## Functions

In [6]:
def decode_coordinate(x, y, z, radius=RADIUS_MM, n_top=N_TOP_TERMS):
    """Return top Neurosynth terms for a coordinate.

    Parameters
    ----------
    x, y, z : int or float
        MNI coordinates in mm.
    radius : float
        Search radius in mm.
    n_top : int
        Number of top terms to return.

    Returns
    -------
    pd.DataFrame with columns: term, z_score, p_active, p_global, n_studies
    """
    dists = np.sqrt(
        (coords_df['x'] - x)**2 +
        (coords_df['y'] - y)**2 +
        (coords_df['z'] - z)**2
    )
    nearby_ids = set(coords_df.loc[dists <= radius, 'id'].unique())
    active_idx = [id_to_idx[sid] for sid in nearby_ids if sid in id_to_idx]
    n_active = len(active_idx)

    if n_active == 0:
        print(f'  Warning: no studies found within {radius}mm of ({x},{y},{z})')
        return pd.DataFrame()

    mask = np.zeros(len(study_ids), dtype=bool)
    mask[active_idx] = True

    p_active = feat_bin[mask].mean(axis=0)
    se = np.sqrt(p_term_global * (1 - p_term_global) / n_active + 1e-10)
    z_scores = (p_active - p_term_global) / se

    top_idx = np.argsort(z_scores)[::-1][:n_top]
    return pd.DataFrame({
        'term':      [vocab[i] for i in top_idx],
        'z_score':   np.round(z_scores[top_idx], 2),
        'p_active':  np.round(p_active[top_idx], 3),
        'p_global':  np.round(p_term_global[top_idx], 3),
        'n_studies': n_active,
    })


def decode_and_display(coords):
    """Decode a list of (x, y, z, label) tuples and display results."""
    results = {}
    for x, y, z, label in coords:
        df = decode_coordinate(x, y, z)
        results[(x, y, z, label)] = df
        print(f'\n━━━ ({x}, {y}, {z})  {label}  [n={df["n_studies"].iloc[0]} studies] ━━━')
        display(df[['term', 'z_score', 'p_active', 'p_global']].reset_index(drop=True))
    return results

---
## PPI — whole-brain clusters (FWE cluster-corrected, p < 0.05)

Seed: posterior putamen (from GLM2 chosen H-value peak). Modulator: chosen H-values at second stimulus onset.

9 clusters survive whole-brain FWE correction (forming threshold p < 0.001). Decoding only the 5 retained for interpretation (bold in LH - PPI note); occipital/cerebellar clusters excluded.

In [7]:
ppi_clusters = [
    (-24, -21,  -8,  'Hippocampus'),
    (-33,  33,  -1,  'Insula / IFC'),
    (-48,  30,  27,  'Middle frontal gyrus (L)'),
    (-48, -30,  37.5,'Postcentral gyrus'),
    ( 45,  33,  30.5,'Middle frontal gyrus (R)'),
]

ppi_results = decode_and_display(ppi_clusters)


━━━ (-24, -21, -8)  Hippocampus  [n=1072 studies] ━━━


,term,z_score,p_active,p_global
0,hippocampus,19.64,0.230,0.074
1,hippocampal,14.41,0.125,0.039
2,medial temporal,13.76,0.105,0.032
3,autobiographical,11.18,0.044,0.010
4,lobe mtl,9.95,0.034,0.007
5,mtl,9.56,0.038,0.010
6,retrieval,9.33,0.129,0.061
7,memory,8.81,0.297,0.191
8,episodic,8.53,0.081,0.034
9,encoding,8.43,0.123,0.061



━━━ (-33, 33, -1)  Insula / IFC  [n=1335 studies] ━━━


,term,z_score,p_active,p_global
0,inferior frontal,8.29,0.208,0.132
1,words,6.72,0.112,0.066
2,orthographic,6.52,0.026,0.009
3,frontal,6.06,0.407,0.329
4,inferior,6.04,0.305,0.235
5,semantic,5.86,0.113,0.072
6,phonological,5.82,0.052,0.026
7,language,5.53,0.117,0.077
8,anterior insula,5.38,0.079,0.047
9,insula,5.25,0.190,0.140



━━━ (-48, 30, 27)  Middle frontal gyrus (L)  [n=1395 studies] ━━━


,term,z_score,p_active,p_global
0,memory,11.96,0.317,0.191
1,working memory,11.03,0.154,0.076
2,working,10.83,0.156,0.078
3,dorsolateral,9.40,0.161,0.089
4,prefrontal,9.11,0.419,0.306
5,task,8.27,0.526,0.417
6,memory wm,8.22,0.047,0.018
7,retrieval,8.11,0.113,0.061
8,demands,8.08,0.085,0.041
9,tasks,7.90,0.232,0.155



━━━ (-48, -30, 37.5)  Postcentral gyrus  [n=1171 studies] ━━━


,term,z_score,p_active,p_global
0,motor,11.14,0.303,0.178
1,parietal,11.06,0.434,0.288
2,hand,10.05,0.132,0.061
3,premotor,9.90,0.147,0.072
4,movements,8.86,0.102,0.047
5,action,8.85,0.124,0.062
6,finger,8.41,0.060,0.023
7,actions,8.30,0.094,0.044
8,inferior parietal,8.11,0.145,0.081
9,motor imagery,7.77,0.026,0.007



━━━ (45, 33, 30.5)  Middle frontal gyrus (R)  [n=1386 studies] ━━━


,term,z_score,p_active,p_global
0,working,13.63,0.177,0.078
1,working memory,13.36,0.171,0.076
2,dorsolateral,12.42,0.185,0.089
3,prefrontal,11.75,0.452,0.306
4,dorsolateral prefrontal,11.44,0.153,0.073
5,prefrontal cortex,10.41,0.343,0.226
6,memory,9.39,0.290,0.191
7,task,8.66,0.532,0.417
8,load,8.64,0.062,0.025
9,parietal,8.63,0.392,0.288


---
## GLM3 — chosen choice variable model (supplementary)

Parametric modulator: chosen choice variable (Q+H combination). 4 clusters retained for interpretation
(bold in GLM3 - choice variable note); cerebellum, occipital, and WM clusters excluded.
Whole-brain FWE cluster-corrected (forming threshold p < 0.001, cluster p < 0.05 FWE).

In [8]:
glm3_clusters = [
    (-30, -18,  -1,   'Left Putamen'),
    (-33, -45,  69,   'Postcentral gyrus (L)'),
    ( 39,  -6, -25.5, 'Right Putamen'),
    (  3, -51,  69,   'Postcentral gyrus (R)'),
]

glm3_results = decode_and_display(glm3_clusters)


━━━ (-30, -18, -1)  Left Putamen  [n=780 studies] ━━━


,term,z_score,p_active,p_global
0,basal,8.95,0.100,0.038
1,putamen,8.69,0.104,0.042
2,basal ganglia,8.50,0.088,0.034
3,ganglia,8.46,0.088,0.034
4,hippocampus,6.10,0.131,0.074
5,paced,6.06,0.026,0.007
6,striatal,5.70,0.077,0.038
7,striatum,5.20,0.106,0.062
8,structures,5.08,0.112,0.066
9,insula,4.90,0.201,0.140



━━━ (-33, -45, 69)  Postcentral gyrus (L)  [n=396 studies] ━━━


,term,z_score,p_active,p_global
0,premotor,11.16,0.217,0.072
1,hand,10.02,0.182,0.061
2,motor,8.97,0.351,0.178
3,dorsal premotor,8.59,0.063,0.013
4,limb,8.32,0.048,0.009
5,parietal,8.23,0.475,0.288
6,motor imagery,7.79,0.040,0.007
7,action,7.64,0.154,0.062
8,superior parietal,7.37,0.119,0.043
9,action observation,7.10,0.040,0.008



━━━ (39, -6, -25.5)  Right Putamen  [n=530 studies] ━━━


,term,z_score,p_active,p_global
0,amygdala,12.75,0.283,0.110
1,hippocampus,9.97,0.187,0.074
2,hippocampal,8.74,0.113,0.039
3,temporal lobe,6.36,0.102,0.045
4,faces,6.05,0.123,0.060
5,amygdala anterior,6.05,0.025,0.005
6,medial temporal,5.99,0.077,0.032
7,lateral temporal,5.96,0.032,0.008
8,temporal,5.71,0.430,0.315
9,pictures,5.54,0.085,0.039



━━━ (3, -51, 69)  Postcentral gyrus (R)  [n=315 studies] ━━━


,term,z_score,p_active,p_global
0,precuneus,5.67,0.152,0.071
1,compensate,4.73,0.029,0.007
2,independent component,4.62,0.063,0.024
3,somatosensory cortices,4.40,0.029,0.007
4,somatosensory,4.32,0.098,0.047
5,motor performance,4.10,0.022,0.005
6,navigation,4.10,0.022,0.005
7,central,4.08,0.117,0.062
8,sensory motor,4.06,0.038,0.013
9,gyrus cerebellum,4.01,0.025,0.007


---
## GLM2 chosen — premotor H-value peak

Contrast: chosen H-value at second stimulus onset. Motor mask SVC survives correction.

- **Whole-brain peak**: (24, −45, 62) — postcentral/premotor cluster
- **SVC peak (premotor mask)**: (48, −6, 20) — survives SVC, but 94/98 voxels WM; decoding included for completeness


In [9]:
glm2_chosen_premotor_clusters = [
    (24, -45, 62,  "Postcentral/premotor (whole-brain peak)"),
    (48,  -6, 20,  "Premotor SVC peak (mostly WM — interpret with caution)"),
]

glm2_chosen_premotor_results = decode_and_display(glm2_chosen_premotor_clusters)



━━━ (24, -45, 62)  Postcentral/premotor (whole-brain peak)  [n=580 studies] ━━━


,term,z_score,p_active,p_global
0,parietal,8.74,0.452,0.288
1,movements,8.60,0.122,0.047
2,motor,7.97,0.305,0.178
3,contralateral,7.63,0.086,0.031
4,somatosensory,7.62,0.114,0.047
5,premotor,7.58,0.153,0.072
6,superior parietal,7.32,0.105,0.043
7,hand,6.68,0.128,0.061
8,tactile,6.66,0.050,0.016
9,primary,6.54,0.166,0.088



━━━ (48, -6, 20)  Premotor SVC peak (mostly WM — interpret with caution)  [n=665 studies] ━━━


,term,z_score,p_active,p_global
0,posterior insula,7.74,0.045,0.012
1,somatosensory,6.94,0.104,0.047
2,s1,6.80,0.027,0.006
3,insula,6.44,0.227,0.140
4,postcentral,6.23,0.054,0.020
5,somatosensory cortices,5.93,0.027,0.007
6,production,5.87,0.056,0.022
7,motor,5.20,0.256,0.178
8,sensorimotor cortex,5.01,0.033,0.012
9,postcentral gyrus,5.00,0.035,0.013
